pip install opencv-python mtcnn torch torchvision tf-keras
 mediapipe filterpy numpy


In [ ]:
# alter code mit video po-up

import cv2
import numpy as np
from mtcnn.mtcnn import MTCNN
from deepface import DeepFace
import mediapipe as mp
from filterpy.kalman import KalmanFilter
from deep_sort_realtime.deepsort_tracker import DeepSort
import os

class VideoAnalysisPipeline:
    def __init__(self, source=0, lip_motion_thresh=2.0, output_path="analysis_results.txt"):
        # Video source (0 für Webcam oder Pfad zur Datei)
        self.cap = cv2.VideoCapture(source)
        # Gesichtserkennung & Landmark-Detektion
        self.detector = MTCNN()
        # Tracking mit DeepSORT
        self.tracker = DeepSort(max_age=30)
        # Kalman-Filter für sanfte Zählung der Personenanzahl
        self.kf = KalmanFilter(dim_x=1, dim_z=1)
        self.kf.x = np.array([[0.]])
        self.kf.F = np.array([[1.]])
        self.kf.H = np.array([[1.]])
        # Schwellenwert für Lippenbewegung zur Sprecher-Erkennung
        self.lip_motion_thresh = lip_motion_thresh
        self.prev_lip_positions = {}
        self.current_speaker_id = None
        # Speicher für alle Frame-Ergebnisse
        self.results = []
        # Ausgabe-Datei
        self.output_path = output_path

    def preprocess(self, frame):
        frame = cv2.fastNlMeansDenoisingColored(frame, None, 10, 10, 7, 21)
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        cl = clahe.apply(l)
        return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)

    def detect_faces(self, frame):
        results = self.detector.detect_faces(frame)
        faces = []
        for res in results:
            x, y, w, h = res['box']
            keypoints = res['keypoints']
            roi = frame[y:y+h, x:x+w]
            faces.append({'box': (x, y, w, h), 'keypoints': keypoints, 'roi': roi})
        return faces

    def classify_gender(self, face):
        try:
            attrs = DeepFace.analyze(face['roi'], actions=['gender'], enforce_detection=False)
            return attrs.get('gender')
        except:
            return None

    def compute_lip_motion(self, face, tid):
        pts = face['keypoints']
        lip_ctr = (np.array(pts['mouth_left']) + np.array(pts['mouth_right'])) / 2
        dist = np.linalg.norm(lip_ctr - self.prev_lip_positions.get(tid, lip_ctr))
        self.prev_lip_positions[tid] = lip_ctr
        return dist

    def analyze_frame(self, frame):
        pre = self.preprocess(frame)
        faces = self.detect_faces(pre)

        # Prepare detections for DeepSORT: (bbox, confidence, class)
        dets = [([x, y, w, h], 1.0, 'face') for (x, y, w, h) in [f['box'] for f in faces]]

        # Tracking update
        tracks = self.tracker.update_tracks(dets, frame=frame)

        # Smoothed person count
        confirmed_tracks = [tr for tr in tracks if tr.is_confirmed()]
        count = len(confirmed_tracks)
        self.kf.predict()
        self.kf.update([[count]])
        smoothed_count = int(self.kf.x[0,0])

        persons = []
        for tr in confirmed_tracks:
            tid = tr.track_id
            x1, y1, x2, y2 = map(int, tr.to_ltrb())
            # Match track to detection
            for f in faces:
                x, y, w, h = f['box']
                if abs(x - x1) < 10 and abs(y - y1) < 10:
                    gender = self.classify_gender(f)
                    lip_motion = self.compute_lip_motion(f, tid)
                    speaking = lip_motion > self.lip_motion_thresh
                    persons.append({'id': tid, 'gender': gender, 'speaking': speaking})
                    break

        # Determine speaker offscreen status
        speakers = [p for p in persons if p['speaking']]
        offscreen = False
        if speakers:
            self.current_speaker_id = speakers[0]['id']
        else:
            offscreen = self.current_speaker_id not in [p['id'] for p in persons]

        return {
            'timestamp_ms': int(self.cap.get(cv2.CAP_PROP_POS_MSEC)),
            'person_count': smoothed_count,
            'speaker_id': self.current_speaker_id,
            'speaker_offscreen': offscreen,
            'persons': persons
        }

    def run(self):
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break

            info = self.analyze_frame(frame)

            # Draw on frame
            cv2.putText(frame, f"Persons: {info['person_count']}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            if info['speaker_id'] is not None:
                status = 'Offscreen' if info['speaker_offscreen'] else 'Onscreen'
                cv2.putText(frame, f"Speaker ID: {info['speaker_id']} ({status})", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            cv2.imshow('Video Analysis', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            # Append result
            self.results.append(info)

        # Cleanup
        self.cap.release()
        cv2.destroyAllWindows()

        # Write to TXT file
        os.makedirs(os.path.dirname(self.output_path) or '.', exist_ok=True)
        with open(self.output_path, 'w', encoding='utf-8') as f:
            for row in self.results:
                persons_str = ', '.join(
                    [f"id:{p['id']} gender:{p['gender']} speaking:{p['speaking']}" for p in row['persons']]
                )
                line = (
                    f"Time {row['timestamp_ms']}ms | Count: {row['person_count']} | "
                    f"Speaker: {row['speaker_id']} ({'Offscreen' if row['speaker_offscreen'] else 'Onscreen'}) | "
                    f"Persons: {persons_str}\n"
                )
                f.write(line)
        print(f"Ergebnisse gespeichert in: {self.output_path}")

if __name__ == '__main__':
    pipeline = VideoAnalysisPipeline(
        source="Diskusion_mit_kommentaren.mkv",
        lip_motion_thresh=6.0,
        output_path="analysis_results.txt"
    )
    pipeline.run()


In [ ]:
# noch nicht "verschnellerter" code


import cv2
import numpy as np
from mtcnn.mtcnn import MTCNN
from deepface import DeepFace
import mediapipe as mp
from filterpy.kalman import KalmanFilter
from deep_sort_realtime.deepsort_tracker import DeepSort
import os

class VideoAnalysisPipeline:
    def __init__(self, source=0, lip_motion_thresh=2.0, output_path="analysis_results.txt"):
        # Video source (0 für Webcam oder Pfad zur Datei)
        self.cap = cv2.VideoCapture(source)
        # Gesichtserkennung & Landmark-Detektion
        self.detector = MTCNN()
        # Tracking mit DeepSORT
        self.tracker = DeepSort(max_age=30)
        # Kalman-Filter für sanfte Zählung der Personenanzahl
        self.kf = KalmanFilter(dim_x=1, dim_z=1)
        self.kf.x = np.array([[0.]])
        self.kf.F = np.array([[1.]])
        self.kf.H = np.array([[1.]])
        # Schwellenwert für Lippenbewegung zur Sprecher-Erkennung
        self.lip_motion_thresh = lip_motion_thresh
        self.prev_lip_positions = {}
        self.current_speaker_id = None
        # Speicher für alle Frame-Ergebnisse
        self.results = []
        # Ausgabe-Datei
        self.output_path = output_path

    def preprocess(self, frame):
        frame = cv2.fastNlMeansDenoisingColored(frame, None, 10, 10, 7, 21)
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        cl = clahe.apply(l)
        return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)

    def detect_faces(self, frame):
        results = self.detector.detect_faces(frame)
        faces = []
        for res in results:
            x, y, w, h = res['box']
            keypoints = res['keypoints']
            roi = frame[y:y+h, x:x+w]
            faces.append({'box': (x, y, w, h), 'keypoints': keypoints, 'roi': roi})
        return faces

    def classify_gender(self, face):
        try:
            attrs = DeepFace.analyze(face['roi'], actions=['gender'], enforce_detection=False)
            return attrs.get('gender')
        except:
            return None

    def compute_lip_motion(self, face, tid):
        pts = face['keypoints']
        lip_ctr = (np.array(pts['mouth_left']) + np.array(pts['mouth_right'])) / 2
        dist = np.linalg.norm(lip_ctr - self.prev_lip_positions.get(tid, lip_ctr))
        self.prev_lip_positions[tid] = lip_ctr
        return dist

    def analyze_frame(self, frame):
        pre = self.preprocess(frame)
        faces = self.detect_faces(pre)

        # Prepare detections for DeepSORT: (bbox, confidence, class)
        dets = [([x, y, w, h], 1.0, 'face') for (x, y, w, h) in [f['box'] for f in faces]]

        # Tracking update
        tracks = self.tracker.update_tracks(dets, frame=frame)

        # Smoothed person count
        confirmed_tracks = [tr for tr in tracks if tr.is_confirmed()]
        count = len(confirmed_tracks)
        self.kf.predict()
        self.kf.update([[count]])
        smoothed_count = int(self.kf.x[0,0])

        persons = []
        for tr in confirmed_tracks:
            tid = tr.track_id
            x1, y1, x2, y2 = map(int, tr.to_ltrb())
            # Match track to detection
            for f in faces:
                x, y, w, h = f['box']
                if abs(x - x1) < 10 and abs(y - y1) < 10:
                    gender = self.classify_gender(f)
                    lip_motion = self.compute_lip_motion(f, tid)
                    speaking = lip_motion > self.lip_motion_thresh
                    persons.append({'id': tid, 'gender': gender, 'speaking': speaking})
                    break

        # Determine speaker offscreen status
        speakers = [p for p in persons if p['speaking']]
        offscreen = False
        if speakers:
            self.current_speaker_id = speakers[0]['id']
        else:
            offscreen = self.current_speaker_id not in [p['id'] for p in persons]

        return {
            'timestamp_ms': int(self.cap.get(cv2.CAP_PROP_POS_MSEC)),
            'person_count': smoothed_count,
            'speaker_id': self.current_speaker_id,
            'speaker_offscreen': offscreen,
            'persons': persons
        }

    def run(self):
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break

            info = self.analyze_frame(frame)

            # Draw on frame
            cv2.putText(frame, f"Persons: {info['person_count']}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            if info['speaker_id'] is not None:
                status = 'Offscreen' if info['speaker_offscreen'] else 'Onscreen'
                cv2.putText(frame, f"Speaker ID: {info['speaker_id']} ({status})", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            cv2.imshow('Video Analysis', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            # Append result
            self.results.append(info)

        # Cleanup
        self.cap.release()
        cv2.destroyAllWindows()

        # Write to TXT file
        os.makedirs(os.path.dirname(self.output_path) or '.', exist_ok=True)
        with open(self.output_path, 'w', encoding='utf-8') as f:
            for row in self.results:
                persons_str = ', '.join(
                    [f"id:{p['id']} gender:{p['gender']} speaking:{p['speaking']}" for p in row['persons']]
                )
                line = (
                    f"Time {row['timestamp_ms']}ms | Count: {row['person_count']} | "
                    f"Speaker: {row['speaker_id']} ({'Offscreen' if row['speaker_offscreen'] else 'Onscreen'}) | "
                    f"Persons: {persons_str}\n"
                )
                f.write(line)
        print(f"Ergebnisse gespeichert in: {self.output_path}")

if __name__ == '__main__':
    pipeline = VideoAnalysisPipeline(
        source="lbak1s.mpg",
        lip_motion_thresh=2.0,
        output_path="analysis_results.txt"
    )
    pipeline.run()


In [ ]:
# "schnellerer" code

import cv2
import numpy as np
from mtcnn.mtcnn import MTCNN
from deepface import DeepFace
from filterpy.kalman import KalmanFilter
from deep_sort_realtime.deepsort_tracker import DeepSort
import os

class VideoAnalysisPipeline:
    def __init__(
        self,
        source=0,
        lip_motion_thresh=2.0,
        output_path="analysis_results.txt",
        scale_factor=0.5,
        skip_frames=2,
    ):
        # Video source
        self.cap = cv2.VideoCapture(source)
        # Downscale and skip settings
        self.scale = scale_factor
        self.skip = skip_frames
        self.frame_count = 0

        # Gesichtserkennung (MTCNN) – auf kleiner Auflösung
        self.detector = MTCNN()
        # Tracking mit DeepSORT (mobilenet-Embedder für Geschwindigkeit)
        self.tracker = DeepSort(
            max_age=30,
            embedder="mobilenet",
            half=True,
            bgr=True,
        )
        # Kalman-Filter für Personenzählung
        self.kf = KalmanFilter(dim_x=1, dim_z=1)
        self.kf.x = np.array([[0.]])
        self.kf.F = np.array([[1.]])
        self.kf.H = np.array([[1.]])

        # Lippenbewegungs-Threshold
        self.lip_motion_thresh = lip_motion_thresh
        self.prev_lip_positions = {}
        self.current_speaker_id = None

        # Ergebnis-Speicher
        self.results = []
        self.output_path = output_path

        # Caches
        self.gender_cache = {}

    def preprocess(self, frame):
        # Downscale
        small = cv2.resize(
            frame, (0,0), fx=self.scale, fy=self.scale,
            interpolation=cv2.INTER_LINEAR
        )
        # Denoise + CLAHE
        small = cv2.fastNlMeansDenoisingColored(small, None, 10,10,7,21)
        lab = cv2.cvtColor(small, cv2.COLOR_BGR2LAB)
        l,a,b = cv2.split(lab)
        cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
        small = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)
        return small

    def detect_faces(self, frame):
        # Detection on downscaled frame
        boxes = []
        keypts = []
        results = self.detector.detect_faces(frame)
        for res in results:
            x,y,w,h = res['box']
            boxes.append((int(x/self.scale), int(y/self.scale),
                          int(w/self.scale), int(h/self.scale)))
            keypts.append(res['keypoints'])
        return boxes, keypts

    def classify_gender(self, roi, tid):
        # Only classify once per track
        if tid not in self.gender_cache:
            try:
                attrs = DeepFace.analyze(
                    roi, actions=['gender'], enforce_detection=False
                )
                self.gender_cache[tid] = attrs.get('gender')
            except:
                self.gender_cache[tid] = None
        return self.gender_cache[tid]

    def compute_lip_motion(self, keypoints, tid):
        lip_ctr = (np.array(keypoints['mouth_left'])
                   + np.array(keypoints['mouth_right'])) / 2
        prev = self.prev_lip_positions.get(tid, lip_ctr)
        dist = np.linalg.norm(lip_ctr - prev)
        self.prev_lip_positions[tid] = lip_ctr
        return dist

    def analyze_frame(self, frame):
        self.frame_count += 1
        if self.frame_count % self.skip != 0:
            # Reuse last results for skipped frames
            return self.results[-1] if self.results else None

        # Resize+preprocess for detection
        small = self.preprocess(frame)
        boxes, keypts = self.detect_faces(small)
        persons = []

        # Prepare detections
        dets = []
        for (x,y,w,h) in boxes:
            dets.append(([x,y,w,h], 1.0, 'face'))

        # Tracking
        tracks = self.tracker.update_tracks(dets, frame=frame)
        confirmed = [t for t in tracks if t.is_confirmed()]

        # Smooth count
        count = len(confirmed)
        self.kf.predict()
        self.kf.update([[count]])
        smoothed = int(self.kf.x[0,0])

        # Map track→face by center proximity
        for tr in confirmed:
            tid = tr.track_id
            x1,y1,x2,y2 = map(int, tr.to_ltrb())
            cx, cy = (x1+x2)//2, (y1+y2)//2
            # find matching face
            for (box, kp) in zip(boxes, keypts):
                bx,by,bw,bh = box
                if bx < cx < bx+bw and by < cy < by+bh:
                    # ROI from full frame
                    roi = frame[by:by+bh, bx:bx+bw]
                    gender = self.classify_gender(roi, tid)
                    lip_motion = self.compute_lip_motion(kp, tid)
                    speaking = lip_motion > self.lip_motion_thresh
                    persons.append({'id':tid, 'gender':gender, 'speaking':speaking})
                    break

        # Speaker logic
        speakers = [p for p in persons if p['speaking']]
        off = False
        if speakers:
            self.current_speaker_id = speakers[0]['id']
        else:
            off = self.current_speaker_id not in [p['id'] for p in persons]

        info = {
            'timestamp_ms': int(self.cap.get(cv2.CAP_PROP_POS_MSEC)),
            'person_count': smoothed,
            'speaker_id': self.current_speaker_id,
            'speaker_offscreen': off,
            'persons': persons
        }
        self.results.append(info)
        return info

    def run(self):
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break
            info = self.analyze_frame(frame)
            if info is None:
                continue
            # (optional) draw on frame
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        self.cap.release()
        cv2.destroyAllWindows()

        # Write TXT
        os.makedirs(os.path.dirname(self.output_path) or '.', exist_ok=True)
        with open(self.output_path, 'w', encoding='utf-8') as f:
            for row in self.results:
                persons_str = ', '.join(
                    [f"id:{p['id']} gender:{p['gender']} speaking:{p['speaking']}"
                     for p in row['persons']]
                )
                line = (
                    f"Time {row['timestamp_ms']}ms | Count: {row['person_count']} | "
                    f"Speaker: {row['speaker_id']} ({'Off' if row['speaker_offscreen'] else 'On'}) | "
                    f"Persons: {persons_str}\n"
                )
                f.write(line)
        print(f"Ergebnisse gespeichert in: {self.output_path}")

if __name__ == '__main__':
    pipeline = VideoAnalysisPipeline(
        source="zwei_personen.mkv",
        lip_motion_thresh=5.0,
        output_path="analysis_results_zwei_personen_fast.txt",
        scale_factor=0.3,
        skip_frames=5
    )
    pipeline.run()

    #s


In [ ]:
# "optimierter" code

import cv2
import numpy as np
import os
from filterpy.kalman import KalmanFilter
from deep_sort_realtime.deepsort_tracker import DeepSort
from concurrent.futures import ThreadPoolExecutor, as_completed

# Optional imports for detection backends
try:
    from mtcnn.mtcnn import MTCNN
except ImportError:
    MTCNN = None
try:
    import mediapipe as mp
    mp_face = mp.solutions.face_detection
except ImportError:
    mp = None
    mp_face = None

# DeepFace for gender
try:
    from deepface import DeepFace
except ImportError:
    DeepFace = None

class VideoAnalysisPipeline:
    def __init__(
        self,
        source=0,
        lip_motion_thresh=2.0,
        output_path="analysis_results.txt",
        enable_downscale=True,
        scale_factor=0.5,
        enable_skip=True,
        skip_frames=2,
        detection_backend='mtcnn',  # 'mtcnn' or 'mediapipe'
        enable_batch_gender=True,
        use_gpu=True,
        enable_multithreading=True,
        max_threads=4
    ):
        # GPU control
        if not use_gpu:
            os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

        # Video source
        self.cap = cv2.VideoCapture(source)
        self.enable_downscale = enable_downscale
        self.scale = scale_factor
        self.enable_skip = enable_skip
        self.skip = skip_frames
        self.frame_count = 0

        # Detection backend
        if detection_backend == 'mtcnn' and MTCNN:
            self.detector = MTCNN()
        elif detection_backend == 'mediapipe' and mp_face:
            self.detector = mp_face.FaceDetection(model_selection=0, min_detection_confidence=0.5)
        else:
            raise ValueError(f"Unknown or unavailable detection backend: {detection_backend}")

        # Tracking with DeepSORT
        self.tracker = DeepSort(
            max_age=30,
            embedder='mobilenet',
            half=True,
            bgr=True,
            embedder_gpu=use_gpu
        )

        # Kalman filter for smoothed count
        self.kf = KalmanFilter(dim_x=1, dim_z=1)
        self.kf.x = np.array([[0.]])
        self.kf.F = np.array([[1.]])
        self.kf.H = np.array([[1.]])

        # Threshold for lip motion
        self.lip_motion_thresh = lip_motion_thresh
        self.prev_lip_positions = {}
        self.current_speaker_id = None

        # Gender cache and batch buffer
        self.gender_cache = {}
        self.batch_buffer = []  # list of (tid, roi)
        self.enable_batch_gender = enable_batch_gender

        # Multithreading
        self.enable_multithreading = enable_multithreading
        self.executor = ThreadPoolExecutor(max_workers=max_threads) if enable_multithreading else None

        # Results storage
        self.results = []
        self.output_path = output_path

    def preprocess(self, frame):
        # Downscale
        if self.enable_downscale:
            frame = cv2.resize(frame, (0,0), fx=self.scale, fy=self.scale)
        # Denoise + CLAHE
        frame = cv2.fastNlMeansDenoisingColored(frame, None, 10,10,7,21)
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l,a,b = cv2.split(lab)
        cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
        return cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)

    def detect_faces(self, frame):
        if isinstance(self.detector, MTCNN):
            res = self.detector.detect_faces(frame)
            boxes, keypts = [], []
            for r in res:
                x,y,w,h = r['box']
                boxes.append((x,y,w,h))
                keypts.append(r['keypoints'])
            return boxes, keypts
        else:
            # MediaPipe
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.detector.process(image_rgb)
            boxes, keypts = [], []
            if results.detections:
                for det in results.detections:
                    loc = det.location_data.relative_bounding_box
                    h,w,_ = frame.shape
                    x,y = int(loc.xmin*w), int(loc.ymin*h)
                    w_box,h_box = int(loc.width*w), int(loc.height*h)
                    boxes.append((x,y,w_box,h_box))
                    keypts.append({})  # no landmarks
            return boxes, keypts

    def classify_gender_batch(self):
        # batch_buffer: list of (tid, roi)
        tids, rois = zip(*self.batch_buffer)
        try:
            results = DeepFace.analyze(
                list(rois), actions=['gender'], enforce_detection=False
            )
        except Exception:
            results = [None]*len(rois)
        for tid, res in zip(tids, results):
            gender = res.get('gender') if res else None
            self.gender_cache[tid] = gender
        self.batch_buffer.clear()

    def compute_lip_motion(self, keypoints, tid):
        pts = keypoints
        if 'mouth_left' in pts and 'mouth_right' in pts:
            lip_ctr = (np.array(pts['mouth_left']) + np.array(pts['mouth_right']))/2
            prev = self.prev_lip_positions.get(tid, lip_ctr)
            dist = np.linalg.norm(lip_ctr - prev)
            self.prev_lip_positions[tid] = lip_ctr
            return dist
        return 0.0

    def analyze_frame(self, frame):
        self.frame_count += 1
        if self.enable_skip and self.frame_count % self.skip != 0:
            return self.results[-1] if self.results else None

        # Preprocess small for detection
        small = self.preprocess(frame)
        boxes, keypts = self.detect_faces(small)
        # Scale boxes back
        if self.enable_downscale:
            boxes = [(
                int(x/self.scale), int(y/self.scale),
                int(w/self.scale), int(h/self.scale)
            ) for x,y,w,h in boxes]

        # Prepare detections
        dets = [([x,y,w,h],1.0,'face') for x,y,w,h in boxes]
        tracks = self.tracker.update_tracks(dets, frame=frame)
        confirmed = [t for t in tracks if t.is_confirmed()]

        # Smoothed count
        count = len(confirmed)
        self.kf.predict()
        self.kf.update([[count]])
        smoothed_count = int(self.kf.x[0,0])

        persons = []
        futures = []
        for tr in confirmed:
            tid = tr.track_id
            x1,y1,x2,y2 = map(int, tr.to_ltrb())
            # match detection
            for (box, kp) in zip(boxes, keypts):
                bx,by,bw,bh = box
                if bx< (x1+x2)//2 < bx+bw and by< (y1+y2)//2 < by+bh:
                    roi = frame[by:by+bh, bx:bx+bw]
                    # schedule gender classification
                    if DeepFace and 'gender' not in self.gender_cache:
                        if self.enable_batch_gender:
                            self.batch_buffer.append((tid, roi))
                        else:
                            self.gender_cache[tid] = DeepFace.analyze(
                                roi, actions=['gender'], enforce_detection=False
                            ).get('gender')
                    # schedule lip_motion and append
                    if self.enable_multithreading:
                        futures.append(self.executor.submit(
                            lambda kp=kp, tid=tid: (
                                tid,
                                self.compute_lip_motion(kp, tid)
                            )
                        ))
                        persons.append({'id':tid, 'gender':None, 'speaking':None, 'kp':kp})
                    else:
                        lip = self.compute_lip_motion(kp, tid)
                        persons.append({'id':tid, 'gender':self.gender_cache.get(tid),
                                        'speaking':lip>self.lip_motion_thresh})
                    break

        # process threaded lip motions
        if self.enable_multithreading and futures:
            for fut, p in zip(as_completed(futures), persons):
                tid, lip = fut.result()
                p['gender'] = p['gender'] or self.gender_cache.get(tid)
                p['speaking'] = lip > self.lip_motion_thresh
                p.pop('kp', None)

        # process gender batch
        if self.enable_batch_gender and self.batch_buffer:
            self.classify_gender_batch()
        # assign gender cache
        for p in persons:
            p['gender'] = p['gender'] or self.gender_cache.get(p['id'])

        # speaker logic
        speakers = [p for p in persons if p['speaking']]
        off = False
        if speakers:
            self.current_speaker_id = speakers[0]['id']
        else:
            off = self.current_speaker_id not in [p['id'] for p in persons]

        info = {
            'timestamp_ms': int(self.cap.get(cv2.CAP_PROP_POS_MSEC)),
            'person_count': smoothed_count,
            'speaker_id': self.current_speaker_id,
            'speaker_offscreen': off,
            'persons': persons
        }
        self.results.append(info)
        return info

    def run(self):
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break
            info = self.analyze_frame(frame)
            if info is None:
                continue
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        self.cap.release()
        cv2.destroyAllWindows()

        # write TXT
        os.makedirs(os.path.dirname(self.output_path) or '.', exist_ok=True)
        with open(self.output_path, 'w', encoding='utf-8') as f:
            for row in self.results:
                persons_str = ', '.join(
                    [f"id:{p['id']} gender:{p['gender']} speaking:{p['speaking']}"
                     for p in row['persons']]
                )
                f.write(
                    f"Time {row['timestamp_ms']}ms | Count: {row['person_count']} | "
                    f"Speaker: {row['speaker_id']} ({'Off' if row['speaker_offscreen'] else 'On'}) | "
                    f"Persons: {persons_str}\n"
                )
        print(f"Ergebnisse gespeichert in: {self.output_path}")

if __name__ == '__main__':
    pipeline = VideoAnalysisPipeline(
        source="zwei_personen.mkv",
        lip_motion_thresh=2.0,
        output_path="analysis_results_zwei_personen_optimiert.txt",
        enable_downscale=True,
        scale_factor=0.3,
        enable_skip=True,
        skip_frames=4,
        detection_backend='mediapipe',
        enable_batch_gender=True,
        use_gpu=False,
        enable_multithreading=True,
        max_threads=4
    )
    pipeline.run()
